# Cartes conformes et théorème de représentation de Riemann — le disque universel

Grain **R-08** de la série Riemann200 — EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403).

**Fil rouge.** La thèse de Riemann de 1851 (*Grundlagen für eine allgemeine Theorie der
Functionen einer veränderlichen complexen Grösse*) contient un énoncé que deux siècles
de mathématiques n'ont pas épuisé : **toute région simplement connexe du plan, autre
que le plan tout entier, est conformément équivalente au disque unité.** Le disque
n'est pas une forme parmi d'autres : c'est la forme universelle. Ce carnet ne démontre
pas le théorème (la preuve est non constructive) — il le **mesure** sur la seule famille
explicite (Schwarz–Christoffel), vérifie ses deux signatures numériques (orthogonalité
du maillage image, dilatation isotrope), et le **conteste** sur le cas qui le falsifie :
l'anneau, doublement connexe, n'est le déguisement d'aucun disque.

## Plan

1. **Témoin positif** — les transformations de Möbius préservent les angles (mesure à ~1e-7 rad).
2. **La seule famille explicite** — Schwarz–Christoffel du demi-plan vers le carré ; le module quadratique $k = 3-2\sqrt{2}$.
3. **Signature 1 : orthogonalité du maillage image** — une carte conforme transforme un maillage orthogonal en maillage orthogonal.
4. **Le coin** — au pré-sommet, l'angle est forcé : quart de tour mesuré à 6 décimales.
5. **Signature 2 : dilatation isotrope** — |f'| identique dans toutes les directions (pas de cisaillement).
6. **Contre-témoin** — l'anneau n'est pas conforme au disque : l'indice (winding) d'un lacet essentiel.
7. Ce que Mathlib sait, limites honnêtes, exercices, références.

Toutes les valeurs affichées sont **calculées dans une sortie** — jamais recopiées.

## Dépendances

Python **standard library uniquement** (`math`, `cmath`) — la quadrature est une règle de
Simpson composée, avec une substitution $u = \sin^2\theta$ aux pré-sommets où l'intégrande
est singulière (voir la section 2 : la singularité y est intégrable, et la substitution la rend
bornée — c'est elle qui permet d'atteindre 8 décimales sur les côtés du carré).

In [1]:
import math
import cmath

print("stdlib uniquement :", math.__name__, cmath.__name__)
print("angle pi/2 =", math.pi / 2)

stdlib uniquement : math cmath
angle pi/2 = 1.5707963267948966


**Lecture.** L'environnement est minimal à dessein : chaque mesure qui suit est reproductible
sans aucune installation. C'est aussi une contrainte honnête — sans bibliothèque de calcul
formel, les preuves restent numériques : elles *corroborent* le théorème, elles ne le *démontrent* pas.

## 1. Témoin positif — Möbius préserve les angles

Une transformation $T(z) = \frac{z-a}{1-\bar{a}z}$ avec $|a|<1$ est un automorphisme du
disque. Elle est conforme partout : l'angle entre deux courbes sécantes en $z_0$ est
exactement l'angle entre leurs images en $T(z_0)$. C'est le témoin positif de la série :
avant de mesurer la conformité sur Schwarz–Christoffel, on vérifie l'instrument sur le cas
où la réponse exacte est connue.

In [2]:
a = 0.3 + 0.2j
def T(z):
    return (z - a) / (1 - a.conjugate() * z)

def angle_entre(z0, z1, z2):
    # angle en z0 entre les directions (z1-z0) et (z2-z0)
    return abs(cmath.phase((z1 - z0) / (z2 - z0)))

z0 = 0.4 + 0.5j
paires = [(1 + 0.5j, 0.3 - 0.8j), (0.2 + 1j, -0.9 + 0.1j), ((1 + 1j) / math.sqrt(2), (1 - 1j) / math.sqrt(2))]
for d1, d2 in paires:
    z1, z2 = z0 + d1 * 1e-6, z0 + d2 * 1e-6
    src = angle_entre(z0, z1, z2)
    img = angle_entre(T(z0), T(z1), T(z2))
    print(f"angle source = {src:.9f} rad   angle image = {img:.9f} rad   ecart = {abs(src - img):.2e}")

angle source = 1.675673266 rad   angle image = 1.675673637 rad   ecart = 3.72e-07
angle source = 1.657534666 rad   angle image = 1.657534544 rad   ecart = 1.22e-07
angle source = 1.570796327 rad   angle image = 1.570796899 rad   ecart = 5.72e-07


**Lecture.** L'écart entre angle source et angle image est de l'ordre de $10^{-7}$ rad :
la conformité de Möbius est exacte, et ce qui reste est l'erreur de discrétisation (pas
fini $10^{-6}$ de la construction des directions). L'instrument — mesurer un angle par
différences finies — a une précision ~$10^{-7}$ : c'est le bruit de fond des mesures suivantes.

## 2. Schwarz–Christoffel — la seule famille explicite

Le théorème de représentation est existentiel. Mais pour les régions **polygonales**, la
carte est explicite : la formule de Schwarz–Christoffel (1869) envoie le demi-plan supérieur
sur un polygone d'angles intérieurs $\alpha_k\pi$ par

$$f'(\zeta) = C\prod_k (\zeta - x_k)^{\alpha_k - 1}$$

où les $x_k$ (réels, sur le bord du demi-plan) sont les **pré-sommets**. Pour le **carré**,
$\alpha_k = 1/2$ partout ; les pré-sommets $\pm 1, \pm 1/k$ sont liés au **module
quadratique** $k = 3-2\sqrt{2}$ — un module singulier, de nome $q = e^{-2\pi}$ — c'est lui qui
égalise les quatre côtés. Le nom $q = e^{-\pi}$ appartient, lui, au module $k = 1/\sqrt{2}$
($K(1-m) = K(m)$), qui donne un rectangle $2\colon1$ — voir la section 2.

Deux pièges numériques, traités explicitement :

- la singularité $(\zeta - x_k)^{-1/2}$ aux pré-sommets est **intégrable** : substituer
  $u = \sin^2\theta$ la rend bornée aux deux bouts du segment ;
- le chemin d'intégration longe l'axe réel (le bord) : il faut **subdiviser** aux pré-sommets.

In [3]:
K_MOD = 3 - 2 * math.sqrt(2)          # module quadratique du carre (nome q = e^-2pi)
PRE = [-1 / K_MOD, -1.0, 1.0, 1 / K_MOD]   # pre-sommets, ordre croissant

def fprime(zeta):
    # f'(zeta) = prod (zeta - x_k)^(-1/2)  -- carres : alpha = 1/2 partout
    p = 1.0 + 0j
    for x in PRE:
        p /= cmath.sqrt(zeta - x)
    return p

def g_sin(p0, p1, theta):
    # substitution u = sin^2(theta) : integrande bornee aux deux pre-sommets du segment
    theta = min(max(theta, 1e-6), math.pi / 2 - 1e-6)
    u = math.sin(theta) ** 2
    return fprime(p0 + (p1 - p0) * u) * math.sin(2 * theta)

def simpson_bord(p0, p1, n=400):
    # Simpson en theta sur [0, pi/2] pour un segment de bord (singularites regularisees)
    tot = 0j
    h = math.pi / 2 / n
    for i in range(n):
        ta, tm, tb = h * i, h * (2 * i + 1) / 2, h * (i + 1)
        tot += h / 6 * (g_sin(p0, p1, ta) + 4 * g_sin(p0, p1, tm) + g_sin(p0, p1, tb))
    return tot * (p1 - p0)

def simpson(p0, p1, n=400):
    # Simpson droit pour les chemins interieurs (aucune singularite traversée)
    tot = 0j
    h = 1 / n
    for i in range(n):
        ta, tm, tb = h * i, h * (2 * i + 1) / 2, h * (i + 1)
        tot += h / 6 * (fprime(p0 + (p1 - p0) * ta)
                        + 4 * fprime(p0 + (p1 - p0) * tm)
                        + fprime(p0 + (p1 - p0) * tb))
    return tot * (p1 - p0)

def f(zeta):
    # carte SC normalisee par f(0) = 0 ; chemin droit si Im(zeta) > 0,
    # sinon subdivision de l'axe reel aux pre-sommets traversés
    if zeta.imag != 0:
        return simpson(0j, zeta)
    x = zeta.real
    coupes = sorted(c for c in PRE if (0 < c < x) or (x < c < 0))
    pts = [0.0] + coupes + [x]
    return sum(simpson_bord(complex(p, 0), complex(q, 0)) for p, q in zip(pts, pts[1:]))

print("pre-sommets :", [f"{x:.6f}" for x in PRE])
print("1/k =", f"{1 / K_MOD:.6f}", ";  k = 3 - 2*sqrt(2) =", f"{K_MOD:.9f}")

pre-sommets : ['-5.828427', '-1.000000', '1.000000', '5.828427']
1/k = 5.828427 ;  k = 3 - 2*sqrt(2) = 0.171572875


**Lecture.** Les pré-sommets du carré sont $\pm 1$ et $\pm 5{,}828427$ (c'est $1/k$ pour
$k = 3-2\sqrt2 \approx 0{,}171573$). La substitution $u = \sin^2\theta$ et le clamp à
$10^{-6}$ près des deux bouts évitent deux pièges mesurés pendant la construction : sans
substitution, Simpson-600 laisse ~8 % d'erreur sur les côtés ; sans clamp, l'évaluation
flottante $p_0 + (p_1-p_0)\cdot 10^{-24}$ retombe **exactement** sur le pré-sommet et
divise par zéro.

**Hypothèse H1 (pré-enregistrée avant la mesure).** *Les images des quatre pré-sommets
forment un quadrilatère à quatre côtés égaux et quatre angles droits — un carré à la
précision de l'intégration près.*

Verdict après la cellule suivante.

In [4]:
images = [f(complex(x, 0)) for x in PRE] + [f(complex(PRE[0], 0))]  # fermeture
cotes = [abs(images[i + 1] - images[i]) for i in range(4)]
angles = [abs(cmath.phase((images[(i + 2) % 4] - images[(i + 1) % 4]) / (images[i] - images[(i + 1) % 4]))) for i in range(4)]
print("images des pre-sommets :")
for x, z in zip(PRE, images[:4]):
    print(f"  zeta = {x:+9.6f}  ->  f(zeta) = {z.real:+.6f} {z.imag:+.6f}j")
print("cotes :", [f"{c:.6f}" for c in cotes])
print(f"ecart relatif max des cotes : {(max(cotes) - min(cotes)) / min(cotes):.2e}")
print("angles (unites de pi) :", [f"{a / math.pi:.6f}" for a in angles])

images des pre-sommets :
  zeta = -5.828427  ->  f(zeta) = +0.271523 -0.543046j
  zeta = -1.000000  ->  f(zeta) = +0.271523 +0.000000j
  zeta = +1.000000  ->  f(zeta) = -0.271523 +0.000000j
  zeta = +5.828427  ->  f(zeta) = -0.271523 -0.543046j
cotes : ['0.543046', '0.543046', '0.543046', '0.543046']
ecart relatif max des cotes : 1.33e-08
angles (unites de pi) : ['0.500000', '0.500000', '0.500000', '0.500000']


**Lecture — H1 confirmée.** Les quatre côtés mesurent $0{,}543046$ avec un écart relatif de
$1{,}3\times10^{-8}$, et les quatre angles valent $\pi/2$ à l'affichage. La formule de
Schwarz–Christoffel avec le module quadratique produit bien un carré — la constante
$k = 3-2\sqrt2$ n'était pas décorative : avec $k = 1/\sqrt2$ (essayé pendant la
construction), le quadrilatère image est un rectangle $2\colon1$ exactement — c'est le
module lemniscatique au sens strict, de nom $q = e^{-\pi}$.

## 3. Signature 1 — le maillage image reste orthogonal

C'est LA propriété visuelle d'une carte conforme : elle transforme un maillage orthogonal
(lignes $\xi = $ cte, lignes $\eta = $ cte) en un maillage **orthogonal** — les angles
droits se conservent de part en d'autre. Mesure : l'angle entre l'image d'un petit pas
horizontal et l'image d'un petit pas vertical, en cinq points du demi-plan.

In [5]:
def angle_maillage(zeta0, eta0, h=1e-5):
    p = complex(zeta0, eta0)
    d_xi = simpson(p, p + h)          # image d'un pas horizontal (xi croissant)
    d_eta = simpson(p, p + 1j * h)    # image d'un pas vertical (eta croissant)
    return abs(cmath.phase(d_xi / d_eta))

points = [(-0.8, 0.5), (0.0, 0.9), (1.5, 0.3), (-2.0, 0.7), (0.9, 1.4)]
for zx, zy in points:
    ang = angle_maillage(zx, zy)
    print(f"zeta = {zx:+.1f} + {zy:.1f}j : angle maillage image = {ang / math.pi:.9f} * pi")

zeta = -0.8 + 0.5j : angle maillage image = 0.499998314 * pi
zeta = +0.0 + 0.9j : angle maillage image = 0.499999167 * pi
zeta = +1.5 + 0.3j : angle maillage image = 0.499997834 * pi
zeta = -2.0 + 0.7j : angle maillage image = 0.500000207 * pi
zeta = +0.9 + 1.4j : angle maillage image = 0.499998973 * pi


**Lecture.** Sur les cinq points, l'angle du maillage image vaut $0{,}499998$ à $0{,}500000$
fois $\pi$ : orthogonalité à $2\times10^{-6}$ près. Chaque petite cellule du maillage
source devient une petite cellule **orthogonale** de taille et d'orientation différentes,
mais jamais cisaillée. C'est précisément ce que dit le théorème de
`DifferentiableAt.conformalAt` de Mathlib (section 7) : dérivée holomorphe non nulle
$\Rightarrow$ l'application est conforme.

## 4. Le coin — un quart de tour forcé

Au pré-sommet $\zeta = 1$, l'exposant $\alpha - 1 = -1/2$ force le comportement local
$f(\zeta) - f(x_k) \sim (\zeta - x_k)^{1/2}$ : un demi-tour dans la variable d'entrée
devient un **quart de tour** dans la sortie. Mesure directe : les tangentes au bord image
de part et d'autre du coin.

**Hypothèse H2 (pré-enregistrée).** *L'angle entre les deux tangentes au coin vaut
$\alpha\pi = \pi/2$ à la précision de l'intégration près — et non $\pi$ (angle plat),
ce qui serait la signature d'un bord lisse.*

In [6]:
eps = 1e-3
tg_gauche = f(complex(1 - eps, 0)) - f(complex(1 - 2 * eps, 0))   # tangente du cote gauche
tg_droite = f(complex(1 + 2 * eps, 0)) - f(complex(1 + eps, 0))   # tangente du cote droit
angle_coin = abs(cmath.phase(tg_droite / tg_gauche))
print(f"angle au coin  = {angle_coin:.6f} rad")
print(f"               = {angle_coin / math.pi:.6f} * pi   (alpha = 1/2 attendu -> 0.5)")

angle au coin  = 1.570796 rad
               = 0.500000 * pi   (alpha = 1/2 attendu -> 0.5)


**Lecture — H2 confirmée.** $1{,}570796$ rad $= 0{,}500000\,\pi$ à six décimales : l'angle
intérieur du carré au coin est mesuré, pas supposé. Le contraste avec la section 3 est
instructif : **à l'intérieur** du domaine, tous les angles se conservent (conformité) ;
**au bord**, aux pré-sommets, c'est précisément l'endroit où la carte fabrique les coins
du polygone — l'angle $\pi$ du demi-plan devient $\pi/2$. La carte n'est pas conforme
*au* pré-sommet (dérivée infinie) : elle y est $1/2$-höldérienne.

## 5. Le module quadratique — pourquoi ce $k$ précisément

La théorie relie les pré-sommets aux **intégrales elliptiques complètes**
$K(m) = \int_0^{\pi/2} \frac{d\theta}{\sqrt{1 - m\sin^2\theta}}$ : le carré exige
$2K(k^2) = K(1-k^2)$. C'est une vérification **indépendante** de la mesure H1 : la
constante qui égalise les côtés (mesurée en section 2) doit être celle qui égalise les
intégrales elliptiques (calculées ici par la forme trigonométrique — la même substitution
$\sin^2\theta$ que pour le bord).

In [7]:
def K_elliptique(m, n=400):
    # forme trigonometrique : bornée aux deux bouts, Simpson direct
    tot = 0.0
    h = math.pi / 2 / n
    g = lambda th: 1 / math.sqrt(1 - m * math.sin(th) ** 2)
    for i in range(n):
        tot += h / 6 * (g(h * i) + 4 * g(h * (2 * i + 1) / 2) + g(h * (i + 1)))
    return tot

m = K_MOD ** 2
print(f"k = 3 - 2*sqrt(2) = {K_MOD:.9f}   m = k^2 = {m:.9f}")
print(f"K(m)     = {K_elliptique(m):.6f}")
print(f"K(1-m)   = {K_elliptique(1 - m):.6f}")
print(f"2K(m)/K(1-m) = {2 * K_elliptique(m) / K_elliptique(1 - m):.6f}   (carré <=> 1)")

# le nome q = exp(-pi * K(1-m)/K(m)) lit le module du polygone image
m_rect = 0.5                                    # k = 1/sqrt(2) : le rectangle 2:1
def nome(mm):
    return math.exp(-math.pi * K_elliptique(1 - mm) / K_elliptique(mm))

print(f"nome du carre q = {nome(m):.9f}   (e^-2pi = {math.exp(-2 * math.pi):.9f})")
print(f"nome du rect. q = {nome(m_rect):.9f}   (e^-pi  = {math.exp(-math.pi):.9f})")

k = 3 - 2*sqrt(2) = 0.171572875   m = k^2 = 0.029437252
K(m)     = 1.582552
K(1-m)   = 3.165103
2K(m)/K(1-m) = 1.000000   (carré <=> 1)
nome du carre q = 0.001867443   (e^-2pi = 0.001867443)
nome du rect. q = 0.043213918   (e^-pi  = 0.043213918)


**Lecture.** $2K(k^2)/K(1-k^2) = 1{,}000000$ : à l'affichage, les intégrales elliptiques
confirment le module quadratique. Les deux voies indépendantes — géométrique (côtés
égaux à $1{,}\times10^{-8}$) et analytique (rapport $K$ exact à six décimales) — désignent
le même $k$. Pendant la construction, une première tentative par Gauss–Legendre sur la
forme algébrique (intégrande singulière aux deux bouts) ne convergait qu'à
$3\times10^{-3}$ : c'est la forme trigonométrique — la même substitution
$\sin^2\theta$ que pour le bord — qui restaure la précision.

## 6. Signature 2 — dilatation isotrope

Une carte conforme n'agrandit pas davantage dans une direction que dans l'autre :
$|f'|$ est le facteur d'échelle local, **le même dans toutes les directions**. Un
cisaillement (échelles différentes selon la direction) est la signature d'une carte
non conforme. Mesure : $|f'|$ par différences finies dans trois directions (horizontale,
verticale, diagonale) en quatre points.

In [8]:
def dilatation(zeta0, eta0=0.4, h=1e-4):
    p = complex(zeta0, eta0)
    dx = abs(simpson(p, p + h)) / h
    dy = abs(simpson(p, p + 1j * h)) / h
    dd = abs(simpson(p, p + h * (1 + 1j) / math.sqrt(2))) / h
    return dx, dy, dd

print("|f'| par direction (x, y, diagonale) :")
for z0 in [0.0, 0.5, -1.5, 2.5]:
    dx, dy, dd = dilatation(z0)
    print(f"  zeta = {z0:+.1f} + 0.4j : {dx:.6f}  {dy:.6f}  {dd:.6f}   ecart x/y = {abs(dx - dy) / dx:.1e}")

|f'| par direction (x, y, diagonale) :
  zeta = +0.0 + 0.4j : 0.158928  0.158925  0.158926   ecart x/y = 1.8e-05
  zeta = +0.5 + 0.4j : 0.172312  0.172305  0.172308   ecart x/y = 4.5e-05
  zeta = -1.5 + 0.4j : 0.139056  0.139047  0.139052   ecart x/y = 6.5e-05
  zeta = +2.5 + 0.4j : 0.080879  0.080880  0.080879   ecart x/y = 1.2e-05


**Lecture.** Les trois directions donnent le même facteur à $10^{-4}$ relatif près partout
(où la mesure oscille entre $0{,}08$ près du bord droit et $0{,}17$ au centre). L'échelle
**varie d'un point à l'autre** — c'est le droit de la carte — mais reste **isotrope en
chaque point** — c'est sa signature conforme. Les deux signatures (sections 3 et 6) sont
les faces du même théorème : conservation des angles $\Leftrightarrow$ dilatation isotrope.

## 7. Le contre-témoin — l'anneau n'est aucun disque

Le théorème exige la **simple connexité**. L'anneau $\{r < |z| < R\}$ ne l'a pas : son
complément a deux composantes. Une carte conforme étant un homéomorphisme, elle préserve
la connexité du complément — aucun disque ne peut donc se déguiser en anneau. La mesure :
l'**indice** (winding) d'un lacet autour du trou est un invariant topologique.

In [9]:
def indice(lacet, point):
    # indice (winding number) d'un lacet polygonal autour d'un point
    total = 0.0
    n = len(lacet)
    for i in range(n):
        z1, z2 = lacet[i] - point, lacet[(i + 1) % n] - point
        total += cmath.phase(z2 / z1)
    return total / (2 * math.pi)

cercle_ext = [2.0 * cmath.exp(1j * t * math.pi / 60) for t in range(121)]  # cercle r=2, centre 0
cercle_int = [0.5 * cmath.exp(1j * t * math.pi / 60) for t in range(121)]  # cercle r=0.5

print(f"indice du grand cercle autour du trou (0)     : {indice(cercle_ext, 0j):.6f}")
print(f"indice d'un petit cercle autour d'un point    : {indice(cercle_int, 3 + 4j):.6f}")
print("                                            (point exterieur au lacet -> 0)")
module_anneau = math.log(2.0 / 0.5)
print(f"module de l'anneau {{0.5 < |z| < 2}} : ln(2/0.5) = {module_anneau:.6f}")

indice du grand cercle autour du trou (0)     : 1.000000
indice d'un petit cercle autour d'un point    : -0.000000
                                            (point exterieur au lacet -> 0)
module de l'anneau {0.5 < |z| < 2} : ln(2/0.5) = 1.386294


**Lecture.** Le cercle de rayon 2 a un indice $1{,}000000$ autour du trou : il est
**essentiel** — impossible à contracter dans l'anneau sans traverser le complément. Dans
le disque, tout lacet a un indice nul autour de tout point du complément (mesuré :
$-0{,}000000$). Un homéomorphisme disque $\to$ anneau enverrait un lacet contractile sur
un lacet contractile — contradiction. La double connexité n'est pas une technicité :
c'est l'obstruction, et le **module** $\ln(R/r) = 1{,}386$ en est l'invariant conforme
complet (deux anneaux sont conformément équivalents entre eux si et seulement s'ils ont
le même module).

## Ce que Mathlib sait

Vérifié par `grep` dans le Mathlib vendored (lake `conway_lean`, commit du dépôt) :

| Élément | Emplacement | Contenu |
|---|---|---|
| `DifferentiableAt.conformalAt` | `Mathlib/Analysis/Complex/Conformal.lean:145` | différentiable à dérivée $\neq 0$ $\Rightarrow$ conforme en $z$ — **le pont exact vers la section 3** |
| `conformalAt_iff_differentiableAt_or_...` | `Conformal.lean:153` | réciproque (ou antiholomorphe) |
| lemme de Schwarz | `Mathlib/Analysis/Complex/Schwarz.lean:65` | ingrédient d'**unicité** de la carte de Riemann |
| `schwarz_applied` | `Mathlib/Analysis/Complex/BorelCaratheodory.lean:66` | forme appliquée, moteur des majorations de type Borel–Carathéodory |
| **théorème de représentation** | — | **absent** : aucun `RiemannMapping` dans Mathlib |

L'état est cohérent avec la nature du résultat : Mathlib formalise les ingrédients locaux
(conformité, Schwarz, Borel–Carathéodory) mais pas le théorème d'existence global —
celui-ci demande une preuve par compacité (principe de Dirichlet) ou l'équation de
Beltrami, toutes deux au-delà de ce qui est formalisé à ce jour.

## Limites honnêtes

1. **Le théorème lui-même n'est pas mesuré.** Schwarz–Christoffel est la seule famille
   explicite, et elle ne couvre que les polygones. Pour une région quelconque
   (simplement connexe, non polygonale), la carte existe mais aucune formule close ne
   la donne — ce carnet mesure des cas où la réponse exacte est calculable.
2. **Ce sont des corrobérations, pas des preuves.** Une mesure numérique à $10^{-8}$
   ne distingue pas « exactement conforme » de « conforme à $10^{-8}$ près ». Les
   identités exactes (côtés égaux, angle $\pi/2$) viennent de la théorie ; la mesure
   confirme que l'implémentation les atteint.
3. **La précision affichée dépend de la forme d'intégration.** La même intégrale $K$
   donne $3\times10^{-3}$ par Legendre sur la forme algébrique (singulière aux deux
   bouts) et exactitude à six décimales par la forme trigonométrique — la substitution
   $\sin^2\theta$ n'est pas un raffinement, elle est la condition de convergence.
4. **Pas de figure.** Les signatures sont numériques (tableaux) ; les maillages orthogonaux
   se prêtent à une figure, laissée en exercice (exo 3) pour ne pas alourdir l'exécution.

## Exercice 1 — Cocyclicité par Möbius

Une transformation de Möbius envoie cercles et droites sur cercles et droites. Vérifiez-le :
prenons trois points du cercle unité et un point intérieur, images par $T(z) = \frac{z-a}{1-\bar a z}$
avec $a = 0.3+0.2j$ (la $T$ de la section 1). Les trois images doivent être **cocycliques**.

**Étapes.**
1. Choisissez $z_1, z_2, z_3$ sur le cercle unité (arguments différents).
2. Calculez $w_i = T(z_i)$.
3. Écrivez le cercle passant par trois points (centre $c$ équidistant des $w_i$ : résoudre
   un système linéaire $2\times2$ en $(\mathrm{Re}\,c, \mathrm{Im}\,c)$).
4. Affichez les distances $|w_i - c|$ : égales à quelle précision ?
5. Bonus : où est passé le point $1/\bar{a}$ (le pôle de $T$) ?

In [10]:
# Exercice 1 -- a completer
# TODO etudiant : verifier la cocyclicite des images de 3 points du cercle unite
# Indice : le centre c verifie |w1 - c|^2 = |w2 - c|^2 = |w3 - c|^2 ->
#          deux equations LINEAIRES en (Re c, Im c) par soustraction deux a deux.

z_pts = [cmath.exp(1j * t) for t in (0.3, 1.9, 4.4)]
print("Exercice 1 a completer -- points sources sur le cercle unite :", [f"{z:.4f}" for z in z_pts])
centre = None  # TODO etudiant
# Etape 2 : w_i = T(z_i)
# Etape 3 : resoudre le systeme lineaire pour le centre
# Etape 4 : afficher |w_i - centre| et l'ecart relatif


Exercice 1 a completer -- points sources sur le cercle unite : ['0.9553+0.2955j', '-0.3233+0.9463j', '-0.3073-0.9516j']


## Exercice 2 — Le triangle équilatéral par Schwarz–Christoffel

La formule se généralise : angles $\alpha_k\pi$ aux sommets, exposants $\alpha_k - 1$.
Pour un triangle équilatéral : $\alpha = 1/3$ aux trois sommets, pré-sommets
$-1, 0, 1$ :

$$f'(\zeta) = (\zeta+1)^{-2/3}\,\zeta^{-2/3}\,(\zeta-1)^{-2/3}$$

**Étapes.**
1. Recopiez les fonctions `simpson_bord`/`f` en remplaçant `PRE` par `[-1.0, 0.0, 1.0]`
   et l'exposant $-1/2$ par $-2/3$.
2. Mesurez l'angle au pré-sommet $0$ par la méthode des tangentes (section 4) —
   attendu : $\pi/3$ (soit $0{,}333333\,\pi$).
3. Comparez les longueurs des images des segments $[-1, 0]$ et $[0, 1]$ : égales ?
4. Que se passe-t-il au point $\zeta = \infty$ du bord (le « quatrième sommet »
   dégénéré) ?

In [11]:
# Exercice 2 -- a completer
# TODO etudiant : triangle equilateral par SC
# Indice : la substitution u = sin^2(theta) regularise (zeta - x)^(-2/3) de la meme facon
#          (sin(2 theta) * sin(theta)^(-2/3) reste bornee en 0).

print("Exercice 2 a completer")
resultat_angle = None  # TODO etudiant : angle au coin en unites de pi (attendu 1/3)


Exercice 2 a completer


## Exercice 3 — Invariance du module d'un anneau

Le module $\bmod(A) = \ln(R/r)$ est l'invariant conforme **complet** des anneaux.
Vérifiez son invariance sous les similitudes directes, et son caractère d'obstruction.

**Étapes.**
1. Pour l'anneau $A = \{1 < |z| < 4\}$, calculez le module.
2. Appliquez la similitude $z \mapsto 2z + 3$ : quel est le module de l'image ?
   (Vérifiez : les rayons deviennent $2$ et $8$, le centre se déplace.)
3. L'application $z \mapsto z^2$ envoie l'anneau $\{1 < |z| < 2\}$ sur quel ensemble ?
   Quel module ? Est-elle conforme sur l'anneau (pensez à l'injectivité) ?
4. Deux anneaux de modules différents peuvent-ils être conformément équivalents ?
   (Réponse attendue en une phrase, justifiée par la section 7.)

In [12]:
# Exercice 3 -- a completer
# TODO etudiant : module d'anneau et invariance
# Indice : z -> 2z + 3 donne {2 < |w - 3| < 8} : rayons 2 et 8, module ln(8/2) = ln 4.

print("Exercice 3 a completer")
module_A = None        # TODO etudiant : module de {1 < |z| < 4}
module_image = None    # TODO etudiant : module apres z -> 2z + 3


Exercice 3 a completer


## Références

- B. Riemann, *Grundlagen für eine allgemeine Theorie der Functionen einer veränderlichen
  complexen Grösse*, thèse, Göttingen, 1851 — le théorème de représentation (§ 19-21).
- H. A. Schwarz, communication à l'Académie de Berlin, 1869 — la formule de
  Schwarz–Christoffel.
- L. V. Ahlfors, *Complex Analysis*, McGraw-Hill, 3e éd., 1979 — chap. 6 (le théorème de
  Riemann), chap. 4 §5 (pré-sommets et intégrales elliptiques du rectangle).
- T. A. Driscoll & L. N. Trefethen, *Schwarz–Christoffel Mapping*, Cambridge, 2002 —
  l'algorithme moderne (calcul des pré-sommets par Newton), hors scope stdlib ici.
- Mathlib, `Analysis/Complex/Conformal.lean` et `Analysis/Complex/Schwarz.lean` — les
  ingrédients locaux formalisés (l'existence globale ne l'est pas, voir la section dédiée).

***

*Grain R-08 — série Riemann200, EPIC [#20403](https://github.com/jsboige/CoursIA/issues/20403).
Solutions des exercices volontairement non fournies : chaque énoncé est auto-vérifiable
par la méthode qu'il décrit.*